# L3 · Manual backprop for $L=(wx+b-y)^2$

**One idea:** *downstream = upstream × local*. We hand-compute the forward pass,
then flow gradients backward through each primitive node and check against
direct calculus. Values: $x=3,\;w=2,\;b=1,\;y=10$.

In [1]:
# forward pass through primitive ops:  m=wx, a=m+b, e=a-y, L=e^2
x, w, b, y = 3.0, 2.0, 1.0, 10.0
m = w*x
a = m + b
e = a - y
L = e**2
print(f"m={m}, a={a}, e={e}, L={L}")     # expect 6, 7, -3, 9

m=6.0, a=7.0, e=-3.0, L=9.0


## Backward: downstream = upstream × local
Start with $\bar L = 1$ and apply each node's local gradient.

In [2]:
gL = 1.0
ge = gL * (2*e)          # L = e^2  -> dL/de = 2e
ga = ge * 1.0            # e = a - y
gy = ge * (-1.0)
gm = ga * 1.0            # a = m + b
gb = ga * 1.0
gw = gm * x              # m = w*x  -> dm/dw = x
gx = gm * w              #          -> dm/dx = w
print(f"grad w = {gw}")  # -18
print(f"grad b = {gb}")  # -6
print(f"grad x = {gx}")  # -12
print(f"grad y = {gy}")  #  6

grad w = -18.0
grad b = -6.0
grad x = -12.0
grad y = 6.0


## Check against direct calculus
$\partial L/\partial w = 2(wx+b-y)x$, $\;\partial L/\partial b = 2(wx+b-y)$.

In [3]:
dw = 2*(w*x + b - y)*x
db = 2*(w*x + b - y)
print("dL/dw:", dw, "== backprop", gw, "->", dw == gw)
print("dL/db:", db, "== backprop", gb, "->", db == gb)

dL/dw: -18.0 == backprop -18.0 -> True
dL/db: -6.0 == backprop -6.0 -> True


## One gradient-descent step ($\eta=0.1$)

In [4]:
eta = 0.1
w_new = w - eta*gw
b_new = b - eta*gb
print(f"w: {w} -> {w_new}")      # 3.8
print(f"b: {b} -> {b_new}")      # 1.6
print("new prediction wx+b =", w_new*x + b_new, "(target 10)")   # 13 -> overshoot

w: 2.0 -> 3.8
b: 1.0 -> 1.6
new prediction wx+b = 12.999999999999998 (target 10)


## Takeaway
- Backprop = repeatedly applying **downstream = upstream × local**, right to left.
- It reproduces textbook calculus exactly ($\nabla_w=-18,\ \nabla_b=-6$).
- The gradient gives the **direction**; the learning rate controls **how far** —
  here $\eta=0.1$ overshoots the target.